<a href="https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
# Setup — makes this notebook work both in Colab and locally.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Zeref538/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # local: move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Data found. Ready.")

Working dir: /content/flyrank-ml-internship
Data found. Ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I want to build a system that tells a content editor which pages to fix first.

I picked this lane because of what I saw in Week 1. In notebook 02, my simple hand-written rule ("old page that still gets traffic") actually **beat** a small decision tree when tested on clients the tree had never seen (0.650 vs 0.400 at Precision@20). But the starter pipeline's bigger model (a random forest) **beat** its rule baseline by a lot (0.740 vs 0.240 at Precision@50), using the same honest client-holdout test.

So sometimes the simple rule wins, and sometimes the model wins. My project is about finding out **when a learned ranking is actually better than a simple rule — and for which pages.**

The starter dataset covers this lane fully, and later I can use the big warehouse data to build a stronger label (use the past 90 days to predict the *next* 30 days). I can still change lanes until Week 4 if the data pushes me somewhere else.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**My research question:** *Which pages should an editor review first for a refresh — and does a learned ranking really put more truly-declining pages at the top of the list than a simple rule?*

**The decision:** An editor only has time to review about **50 pages per cycle**. The decision is: *which 50?*

**Who acts, and what do they do:** The editor opens the top pages on my list and decides what to do with each one — refresh it, expand it, prune it, or just keep watching it. **My system sorts; the human decides.**

**What one row means (unit of analysis):** one content page, for one client, with its last-90-days stats.

**The output:** a ranked list of pages, each with a score and a short reason ("stale but still visible", "good rank but weak clicks") so the editor can see *why* it's on the list.

**The cost of a wrong call:** two different mistakes cost different things.
- *False alarm* — flagging a healthy page → an editor wastes hours on a page that was fine.
- *Miss* — not flagging a dying page → it quietly keeps losing traffic.

Editor time is the scarcer resource, so what matters most is that the **top of the list is right**. That's why my success metric is **Precision@50**: of the 50 pages I say to review first, how many really are declining?

**Why use ML at all?** Because the signal is spread across many features at once — position, freshness, traffic, engagement — tangled together in ways one if-statement can't capture. But the simple rule stays in the project as the thing to beat. If the model *can't* beat it under an honest test, that's still a useful answer: "don't over-engineer this."

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
import os
import pandas as pd

# move from work/notebooks/ up to the repo root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how big is the problem?
declining = (df["trend_direction"].str.lower() == "down").mean()
print(f"1) {declining:.1%} of the 30,000 pages are declining "
      f"({int(declining * len(df)):,} pages). No team can review that by hand.")

# Number 2: can the simple rule handle it?
flagged = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"2) The simple rule (not updated in 180+ days AND 500+ impressions) flags only "
      f"{flagged} pages. It can't even fill one 50-page review cycle, while ~16,000 "
      f"declining pages get no ranking at all. So we need a SCORE, not just a filter.")

# Number 3: is our intuition even right about which pages decline?
# (avg_position = 0 means 'no data' in this dataset, so those rows are excluded)
pos = df[df["avg_position"] > 0].groupby(df["trend_direction"].str.lower())["avg_position"].median()
print(f"3) Median Google position: declining pages sit at {pos['down']:.1f}, growing pages "
      f"at {pos['up']:.1f}. Declining pages actually RANK BETTER — decline happens to "
      f"pages that have something to lose. So 'fix the worst-ranked pages first' targets "
      f"the wrong pages. Patterns like this are what a model can learn and a gut rule misses.")

1) 54.2% of the 30,000 pages are declining (16,262 pages). No team can review that by hand.
2) The simple rule (not updated in 180+ days AND 500+ impressions) flags only 17 pages. It can't even fill one 50-page review cycle, while ~16,000 declining pages get no ranking at all. So we need a SCORE, not just a filter.
3) Median Google position: declining pages sit at 11.3, growing pages at 15.3. Declining pages actually RANK BETTER — decline happens to pages that have something to lose. So 'fix the worst-ranked pages first' targets the wrong pages. Patterns like this are what a model can learn and a gut rule misses.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Words I will use:**

- **"Observed"** — I measured this in the data. ("54% of pages in this sample are declining.")
- **"Directional"** — the data points this way, but it's a hint, not proof. ("This *suggests* freshness matters.")
- **"Decision-support"** — my list orders human attention better than another list. ("Under an honest test, my ranking put more truly-declining pages in the top 50 than the rule did.")

**Claims I will never make:**

- **"Refreshing this page will bring its traffic back."** I can't know that without an experiment. My list finds pages *worth looking at*, not pages guaranteed to recover.
- **"I figured out Google's algorithm."** I'm watching outcomes in one dataset, nothing more.
- **Anything that identifies a client.** Only pseudonymized IDs and aggregate numbers, ever.

**One weakness I'm admitting up front:** the starter label ("is this page declining?") comes from `trend_direction`, which is computed from the *same* time window as my features. That makes it a weak stand-in (a proxy) — and it means `trend_direction` and `trend_pct` can **never** be used as features, because they basically contain the answer. The real fix comes later with the warehouse data: features from the past 90 days, label from the *next* 30 days.

**Traps I know about and will test for:** a traffic drop can be seasonality, or one page stealing a sibling page's traffic — not real decline. And the rate columns are ×100 percentages (`ctr = 0.76` means 0.76%, not 76%) — easy to misread in a threshold.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.